<a href="https://colab.research.google.com/github/mariagres07/tutu-club/blob/main/TutuClub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision timm gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 460.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.4/447.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/13

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import gradio as gr
import torch
import torch.nn as nn
import timm
import numpy as np

# Load the state dictionary from the file
file_path = '/content/drive/MyDrive/vit_tiny_finetuned (1).pth'  # Pastikan path ini sesuai

# Create instance of ViT-Tiny model
model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=2)

# Modify the last layer (fully connected)
model.head = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.head.in_features, 2)
)

# Load state dictionary
try:
    state_dict = torch.load(file_path, map_location='cpu', weights_only=True)
    new_state_dict = {k: v for k, v in state_dict.items() if k in model.state_dict()}
    model.load_state_dict(new_state_dict, strict=False)  # Load a subset of weights
except Exception as e:
    print(f"Error: Could not load state dict: {e}")

# Set the model to evaluation mode
model.eval()


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Error: Could not load state dict: [Errno 2] No such file or directory: '/content/drive/MyDrive/vit_tiny_finetuned (1).pth'


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity()


In [4]:
def preprocess_image(image):
    try:
        # Resize to match model input size
        image = image.resize((224, 224))

        # Convert to RGB
        image = image.convert("RGB")

        # Convert to NumPy array and normalize
        image_array = np.array(image)
        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        image_tensor = image_tensor.type(torch.float32)

        print("Image preprocessed successfully.")  # Debug print
        return image_tensor

    except Exception as e:
        print(f"Error during image preprocessing: {e}")  # Print error message
        return None

In [5]:
def classify_image(image):
    try:
        processed_image = preprocess_image(image)
        if processed_image is None:
            return "Error: Could not preprocess image", [0]*2  # Default probabilities if failed

        with torch.no_grad():
            output = model(processed_image)
            probabilities = torch.softmax(output, dim=1).flatten()
            predicted_class = torch.argmax(probabilities).item()

        class_labels = ["Fake", "Real"]  # Sesuaikan berdasarkan kelas yang ada
        return {class_labels[i]: float(probabilities[i]) for i in range(len(probabilities))}
        print(output)

    except Exception as e:
        print(f"Error during prediction: {e}")
        return "Error: An error occurred during prediction", [0]*2


In [6]:
# Create the Gradio interface
iface = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil", label="Upload Image", sources="upload"),
    outputs=gr.Label(num_top_classes=2, label="Predictions"),
    title="Image Classification with ViT-Tiny",
    description="Upload an image to classify it using the fine-tuned ViT-Tiny model."
)

# Launch the interface
iface.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e91ba0dec23e6e677c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
